# Checkpoint 5 · Release v3.0 — "O Delivery" · Versão B
**Computational Thinking with Python · FIAP Rio · Semestre 2**

---

Hoje não tem conteúdo novo: tem **release** — e desta vez ela **conversa com a internet de verdade**. O Pré-Checkpoint rodava a mesa de teste num dublê da API, com respostas salvas. Aqui não: cada consulta de CEP vai ao **ViaCEP**, e a mesa de teste confere o que a API respondeu **hoje**.

Esta é a **versão B**: a mesma v3.0, a mesma rubrica, por outro caminho. Você **assume uma release que já foi ao ar**, conserta o que quebrou e entrega o que ficou faltando.

- **Rode as células de cima para baixo.** Travou? **Runtime ▸ Restart and run all** (Colab) ou **Run ▸ Run All** (PyCharm/Jupyter).
- **Precisa de internet.** No Colab o `requests` já vem instalado; no PyCharm, rode `pip install requests` no terminal antes de começar.
- Quando aparecer `# TODO`, o código é seu — **inclusive quando a célula já vem com código**: aí o trabalho é consertar, não apagar e recomeçar sem ler.
- As células ✅ **CONFIRA** são a mesa de teste da release: todas têm que terminar em *"Tudo certo!"*.


## A release da vez: v3.0 — "O Delivery"

O sistema que **lembra**, **acha o cliente** e **não cai**. A lista é a mesma do Pré-Checkpoint — release é release:

1. **Cardápio em `cardapio_v3.json`** — salvo e carregado do disco; arquivo ausente ou corrompido cai no cardápio-semente, sem drama (Aula 14).
2. **Toda venda gravada em `vendas_v3.csv` com cabeçalho** — e o fechamento do dia lê o **disco**, não a variável (Aulas 12–13).
3. **Endereço de entrega via CEP — na API de verdade** (ViaCEP), com `timeout`, `status` e `except`; um **cache em JSON** para não perguntar duas vezes a mesma coisa; e o endereço à mão quando nada responde (Aulas 16–17).
4. **Nada derruba o sistema** — entrada suja, produto fantasma, estoque insuficiente, arquivo corrompido, rede fora: tudo vira aviso em português, nunca traceback (Aula 11).
5. **Fila de pedidos de delivery em `pedidos_v3.json`** — cliente + itens + endereço + total, com a taxa de entrega (R$ 5,00, grátis **a partir de** R$ 50,00).

## A rubrica

| critério | o que o professor olha |
|---|---|
| **Roda** | *Restart & Run All* do início ao fim, sem célula pulada |
| **Robusto** | nenhuma entrada errada termina em traceback — agora com arquivo **e rede** |
| **Asserts** | todas as células ✅ CONFIRA terminando em "Tudo certo" |
| **PEP 8** | nomes que dizem o que são, constantes em MAIÚSCULAS, funções com docstring |


## O cenário: o caderno de reclamações

A v3.0 da sua lanchonete **foi ao ar na sexta-feira** — montada às pressas por um colega (com "ajuda" de IA) que copiou pedaços das aulas e saiu do time. No sábado, o caderno de reclamações do delivery encheu. Seis anotações, na letra do gerente:

| # | reclamação | o que está por trás |
|---|---|---|
| 1 | *"Abri o cardápio no Bloco de Notas: estava escrito **A\u00e7a\u00ed**. Arrumei à mão, salvei, e o sistema **não abriu mais**."* | JSON salvo ilegível; carregamento que cai com arquivo corrompido |
| 2 | *"O fechamento do dia mostrou **só a última venda**. Pedimos para arrumar — aí ele **quebrou**."* | CSV sobrescrito; cabeçalho lido como número |
| 3 | *"O motoboy foi parar no **endereço errado**. E quando o wi-fi caiu, o sistema **travou e apagou**."* | o "dublê" da aula no lugar da API; requisição sem `timeout`, sem `except`, sem guarda do `erro` |
| 4 | *"O mesmo cliente é consultado **toda vez**. E o CEP da Dona Marta, depois que a internet caiu, **nunca mais funcionou**."* | cache que quebra no segundo dia e guarda o `None` |
| 5 | *"Pedido de **R$ 50,00 cravados** pagou frete. Cliente de **Paraty** recebeu o endereço **', 10 - - Paraty/RJ'**. E a fila de entregas **só tem o último pedido**."* | `>` no lugar de `>=`; CEP de cidade inteira; JSON sobrescrito |
| 6 | *"Quero ver o **sábado inteiro** rodar — com promoção e com o wi-fi caindo no meio."* | a mesa de teste completa |

Cada reclamação é uma etapa, e cada etapa tem a sua mesa de teste. Leia o código herdado antes de mexer: na defesa oral, a pergunta é *"onde estava o bug?"*, não *"o que você digitou?"*. A regra de sempre vale: **uma função chama a outra, nenhuma regra mora em dois lugares.**

▶️ RODE as duas células abaixo: os dados de partida e a bancada da mesa de teste.


In [3]:
# --- dados de partida da v3.0 (fornecidos — não altere os valores) ---
import json
import requests

LANCHONETE = "Provadelli's "   # 👉 só aqui é seu — batize!

TAXA_ENTREGA = 5.00
FRETE_GRATIS = 50.00            # a partir de 50.00 (inclusive), a entrega é grátis

ARQUIVO_CARDAPIO = 'cardapio_v3.json'
ARQUIVO_VENDAS = 'vendas_v3.csv'
ARQUIVO_CACHE = 'cache_cep_v3.json'
ARQUIVO_PEDIDOS = 'pedidos_v3.json'

CARDAPIO_SEMENTE = {
    "X-Burguer": 18.50,
    "X-Salada": 20.00,
    "Batata frita": 10.00,
    "Açaí 300ml": 12.00,
    "Suco de laranja": 8.00,
    "Refrigerante lata": 6.50,
    "Água mineral": 4.00,
    "Combo da casa": 29.90,
}

# o estoque guarda só QUANTIDADE — o preço mora no cardápio (uma regra, um lugar)
ESTOQUE_INICIAL = {
    "X-Burguer": 6,
    "X-Salada": 5,
    "Batata frita": 10,
    "Açaí 300ml": 3,
    "Suco de laranja": 8,
    "Refrigerante lata": 24,
    "Água mineral": 12,
    "Combo da casa": 4,
}

print(f'{LANCHONETE} — v3.0 no ar. Caderno de reclamações aberto: 6 anotações.')


Provadelli's  — v3.0 no ar. Caderno de reclamações aberto: 6 anotações.


In [4]:
# --- 🧰 a bancada da mesa de teste (fornecida — não altere) ---
# Os CONFIRA desta release usam duas ferramentas:
#   🕵️ espiao_da_rede(): deixa a requisição ir à internet DE VERDADE e anota cada uma
#      (url, timeout, resposta) — prova que o seu código perguntou à API;
#   🔌 rede_fora(...): simula o cabo puxado, a API lenta (timeout) ou fora do ar (503),
#      sem você precisar desligar o wi-fi.
# Os dois entram por requests.get — por isso, no seu código, chame sempre requests.get(...).
import os
from contextlib import contextmanager


@contextmanager
def espiao_da_rede():
    """Deixa as requisições passarem e anota cada uma em uma lista."""
    chamadas = []
    get_de_verdade = requests.get

    def espiao(*args, **kwargs):
        resposta = get_de_verdade(*args, **kwargs)
        url = args[0] if args else kwargs.get('url')
        chamadas.append({'url': url, 'kwargs': kwargs, 'resposta': resposta})
        return resposta

    requests.get = espiao
    try:
        yield chamadas
    finally:
        requests.get = get_de_verdade


@contextmanager
def rede_fora(problema='cabo'):
    """Simula a rede falhando: 'cabo' (sem conexão), 'lenta' (timeout) ou '503'."""
    tentativas = []
    get_de_verdade = requests.get

    def sabotador(*args, **kwargs):
        url = args[0] if args else kwargs.get('url')
        tentativas.append(url)
        if problema == 'lenta':
            raise requests.Timeout('a API demorou demais (simulado pelo CONFIRA)')
        if problema == '503':
            resposta = requests.Response()
            resposta.status_code = 503
            resposta.url = url
            resposta._content = b'<html><h1>503 Service Unavailable</h1></html>'
            return resposta
        raise requests.ConnectionError('cabo puxado (simulado pelo CONFIRA)')

    requests.get = sabotador
    try:
        yield tentativas
    finally:
        requests.get = get_de_verdade


def apagar(*nomes):
    """Apaga os arquivos indicados, se existirem — a mesa de teste começa do zero."""
    for nome in nomes:
        if os.path.exists(nome):
            os.remove(nome)


# 🌐 a internet está aí?
try:
    teste = requests.get('https://viacep.com.br/ws/01001000/json/', timeout=5)
    REDE_OK = teste.status_code == 200
except requests.RequestException:
    REDE_OK = False

if REDE_OK:
    print('ViaCEP respondeu — a mesa de teste vai à internet de verdade. 🌐')
else:
    print('⚠️  O ViaCEP não respondeu. Confira o wi-fi e rode tudo de novo:')
    print('    sem internet, os CONFIRA das reclamações 3 a 6 não têm como corrigir.')


ViaCEP respondeu — a mesa de teste vai à internet de verdade. 🌐


### Reclamação 1 — "Estava escrito A\u00e7a\u00ed — e depois o sistema não abriu mais" (Aulas 14 + 11)

O colega salvou o cardápio sem `ensure_ascii=False`: o arquivo funciona, mas **gente não lê** `A\u00e7a\u00ed`. O gerente abriu no Bloco de Notas, "arrumou" à mão, apagou uma vírgula sem querer — e o `carregar_cardapio` herdado só sabia lidar com arquivo **ausente**, não com arquivo **corrompido**.

🔧 **Conserte** as duas funções, sem mudar nome nem parâmetros:

- `salvar_cardapio(nome, cardapio)` — grava com `encoding='utf-8'`, `indent=4` e `ensure_ascii=False`: um humano tem que conseguir ler e editar o arquivo.
- `carregar_cardapio(nome)` — lê do disco. Arquivo **ausente** ou **corrompido** (`json.JSONDecodeError`) → um aviso em português **para cada causa** e devolve **uma cópia** do cardápio-semente: `dict(CARDAPIO_SEMENTE)`. Devolver a própria constante é entregar a chave do cofre: quem mexer no cardápio carregado estraga a semente.

As duas últimas linhas da célula (fornecidas) gravam o cardápio do dia e carregam `cardapio` do disco — é **esse** dicionário que as próximas reclamações usam.


In [7]:
# --- código herdado: grava ilegível, e cai com arquivo corrompido ---
# TODO: conserte as duas funções (utf-8 + ensure_ascii=False; dois except; devolva uma CÓPIA)

def salvar_cardapio(nome, cardapio):
    """Grava o cardápio no arquivo JSON indicado de forma legível."""
    with open(nome, 'w', encoding='utf-8') as arq:
        json.dump(cardapio, arq, indent=4, ensure_ascii=False)


def carregar_cardapio(nome):
    """Lê o cardápio do disco; sem arquivo ou se estiver corrompido, devolve uma cópia do semente."""
    try:
        with open(nome, 'r', encoding='utf-8') as arq:
            return json.load(arq)
    except FileNotFoundError:
        print('Cardápio não encontrado — usando o semente.')
        return dict(CARDAPIO_SEMENTE)
    except json.JSONDecodeError:
        print('Cardápio corrompido — usando o semente.')
        return dict(CARDAPIO_SEMENTE)

salvar_cardapio(ARQUIVO_CARDAPIO, CARDAPIO_SEMENTE)
cardapio = carregar_cardapio(ARQUIVO_CARDAPIO)

In [8]:
# ✅ CONFIRA — Reclamação 1
TESTE = 'cardapio_teste_v3.json'
salvar_cardapio(TESTE, CARDAPIO_SEMENTE)
with open(TESTE, 'r', encoding='utf-8') as arq:
    texto = arq.read()
assert 'Açaí 300ml' in texto, "abra o arquivo: o gerente tem que ler 'Açaí 300ml', não 'A\\u00e7a\\u00ed' — ensure_ascii=False (e encoding='utf-8')"
assert texto.count('\n') >= 9, "indent=4: um produto por linha, para um humano conseguir editar"
assert carregar_cardapio(TESTE) == CARDAPIO_SEMENTE, "salvou e carregou: tinha que voltar igual à semente"

with open(TESTE, 'w', encoding='utf-8') as arq:   # 📝 o gerente apagou uma vírgula no Bloco de Notas...
    arq.write('{"X-Burguer": 18.50, "X-Salada": 20.00 "Batata frita": 10.00}')
corrompido = carregar_cardapio(TESTE)
assert corrompido == CARDAPIO_SEMENTE, "arquivo corrompido devolve a semente — sem traceback (json.JSONDecodeError)"

sumido = carregar_cardapio('arquivo_que_nao_existe_v3.json')
assert sumido == CARDAPIO_SEMENTE, "arquivo ausente devolve a semente"
sumido['X-Burguer'] = 0.00
assert CARDAPIO_SEMENTE['X-Burguer'] == 18.50, "mexer no cardápio carregado estragou a CONSTANTE — devolva uma cópia: dict(CARDAPIO_SEMENTE)"

assert cardapio == CARDAPIO_SEMENTE, "o cardápio do dia (carregado do disco) não bate com a semente"
apagar(TESTE)
print("Tudo certo! Reclamação 1 fechada — o gerente lê o arquivo, e o arquivo estragado não derruba ninguém. ✅")


Cardápio corrompido — usando o semente.
Cardápio não encontrado — usando o semente.
Tudo certo! Reclamação 1 fechada — o gerente lê o arquivo, e o arquivo estragado não derruba ninguém. ✅


### Reclamação 2 — "O fechamento mostrou só a última venda — e depois quebrou" (Aulas 12–13 + 11)

O `registrar_venda` herdado abre o CSV com `'w'`: **cada venda apaga as anteriores** — e o cabeçalho junto. Por isso o fechamento "funcionava": só sobrava uma linha, e ela era um número. Quando alguém trocou para `'a'`, o cabeçalho voltou a existir e o `fechar_dia` tentou somar a palavra `valor`:

```
ValueError: could not convert string to float: 'valor'
```

E tem mais: o herdado cai com produto fora do cardápio (`KeyError`) e dá baixa no estoque **antes** de saber se tem.

🔧 **Conserte** `registrar_venda(nome, cardapio, estoque, produto, qtd)` e `fechar_dia(nome)` (o `abrir_turno` está certo — leia para ver o cabeçalho):

1. Produto **fora do cardápio** → aviso + devolve **`None`** (é pedido errado do cliente, o caixa segue).
2. Quantidade **zero ou negativa** → `raise ValueError(...)`; estoque **insuficiente** → `raise ValueError(...)`. Nos dois casos, **nada muda**: nem estoque, nem arquivo.
3. Venda válida: `valor = preço do cardápio × qtd`, desconta do `estoque`, **acrescenta** a linha `produto;qtd;valor` (valor com `:.2f`) e devolve o `valor`.
4. `fechar_dia` **pula o cabeçalho** e soma a coluna `valor` lida do disco.

O preço vem do **cardápio** (o carregado do disco); o estoque só guarda quantidade. Uma regra, um lugar.


In [9]:
# --- código herdado: sobrescreve o arquivo, cai com produto fantasma, dá baixa sem olhar ---
# TODO: conserte registrar_venda e fechar_dia (abrir_turno está certo)

def abrir_turno(nome):
    """Cria o CSV do dia zerado, só com o cabeçalho."""
    with open(nome, 'w', encoding='utf-8') as arq:
        arq.write('produto;qtd;valor\n')


def registrar_venda(nome, cardapio, estoque, produto, qtd):
    """Registra a venda no CSV e devolve o valor, atualizando o estoque."""
    if produto not in cardapio:
        print(f"Aviso: Produto '{produto}' não encontrado no cardápio.")
        return None

    if qtd <= 0:
        raise ValueError("A quantidade vendida deve ser maior que zero.")

    if estoque.get(produto, 0) < qtd:
        raise ValueError(f"Estoque insuficiente para '{produto}'. Disponível: {estoque.get(produto, 0)}.")

    preco = cardapio[produto]
    valor = preco * qtd

    # Atualiza o estoque somente após validações
    estoque[produto] -= qtd

    # Escreve no modo 'a' (append) para preservar vendas anteriores
    with open(nome, 'a', encoding='utf-8') as arq:
        arq.write(f'{produto};{qtd};{valor:.2f}\n')

    return valor


def fechar_dia(nome):
    """Relê o CSV do disco e devolve o faturamento total do dia."""
    total = 0.0
    with open(nome, 'r', encoding='utf-8') as arq:
        linhas = arq.readlines()
        # Pula a primeira linha (cabeçalho)
        for linha in linhas[1:]:
            linha_limpa = linha.strip()
            if linha_limpa:
                partes = linha_limpa.split(';')
                total += float(partes[2])
    return total


In [10]:
# ✅ CONFIRA — Reclamação 2
TESTE = 'vendas_teste_v3.csv'
estoque_teste = dict(ESTOQUE_INICIAL)
abrir_turno(TESTE)
v1 = registrar_venda(TESTE, cardapio, estoque_teste, 'X-Salada', 2)            # 2 × 20.00 = 40.00
v2 = registrar_venda(TESTE, cardapio, estoque_teste, 'Refrigerante lata', 3)   # 3 × 6.50 = 19.50
v_fantasma = registrar_venda(TESTE, cardapio, estoque_teste, 'X-Tudo', 1)      # não existe

recusas = 0
for produto, qtd in [('Açaí 300ml', 5), ('X-Burguer', 0), ('X-Burguer', -2)]:
    try:
        registrar_venda(TESTE, cardapio, estoque_teste, produto, qtd)
    except ValueError as erro:
        print('Recusada ->', erro)
        recusas += 1

assert v1 is not None and abs(v1 - 40.00) < 0.005, f"2 X-Saladas dão 40.00 — o seu deu {v1}"
assert v2 is not None and abs(v2 - 19.50) < 0.005, f"3 refrigerantes dão 19.50 — o seu deu {v2}"
assert v_fantasma is None, f"'X-Tudo' não está no cardápio: devolve None — o seu deu {v_fantasma}"
assert recusas == 3, f"Açaí sem estoque (há 3), quantidade 0 e quantidade -2: três ValueError — o seu levantou {recusas}"
assert estoque_teste['X-Salada'] == 3 and estoque_teste['Refrigerante lata'] == 21, "baixa: X-Salada 5 → 3, Refrigerante 24 → 21"
assert estoque_teste['Açaí 300ml'] == 3, f"venda recusada não dá baixa — o Açaí tinha que continuar em 3, está em {estoque_teste['Açaí 300ml']}"
assert estoque_teste['X-Burguer'] == 6, f"nem para baixo, nem para CIMA: o -2 não pode devolver estoque — X-Burguer está em {estoque_teste['X-Burguer']}"

with open(TESTE, 'r', encoding='utf-8') as arq:
    linhas = [linha.strip() for linha in arq]
assert linhas[0] == 'produto;qtd;valor', f"a primeira linha é o cabeçalho — a sua é {linhas[0]!r} (o 'w' apagou o arquivo?)"
assert linhas[1:] == ['X-Salada;2;40.00', 'Refrigerante lata;3;19.50'], f"cabeçalho + 2 vendas boas, valor com 2 casas — o arquivo tem {linhas[1:]}"
faturamento = fechar_dia(TESTE)
assert abs(faturamento - 59.50) < 0.005, f"40.00 + 19.50 = 59.50 — o seu fechou em {faturamento:.2f}"
apagar(TESTE)
print("Tudo certo! Reclamação 2 fechada — o arquivo lembra todas as vendas e o fechamento lê o disco. ✅")


Aviso: Produto 'X-Tudo' não encontrado no cardápio.
Recusada -> Estoque insuficiente para 'Açaí 300ml'. Disponível: 3.
Recusada -> A quantidade vendida deve ser maior que zero.
Recusada -> A quantidade vendida deve ser maior que zero.
Tudo certo! Reclamação 2 fechada — o arquivo lembra todas as vendas e o fechamento lê o disco. ✅


### Reclamação 3 — "O motoboy foi parar no endereço errado — e o sistema travou quando o wi-fi caiu" (Aulas 16–17)

O coração da v3.0. O colega copiou o **dublê** da aula — as `RESPOSTAS_SALVAS` — "pra economizar internet". Só que endereço salvo **envelhece**: a API de hoje responde outra coisa para o `20040-020` (rode a mesa e veja o que ela diz). E quando o CEP não está no dicionário, o herdado vai à rede **sem `timeout`** (com o wi-fi ruim, trava), **sem `except`** (cai) e **sem olhar a chave `erro`** (o ViaCEP responde **200** para CEP inexistente — lembra? *200 quer dizer "a conversa deu certo", não "o dado existe"*).

🔧 **Escreva** `limpar_cep` e **reescreva** `consultar_cep` — e **apague** as `RESPOSTAS_SALVAS`:

- `limpar_cep(texto)` — tira `-`, `.` e espaços das pontas; se sobrarem **8 dígitos**, devolve o CEP limpo (`'20040-020'` → `'20040020'`); senão, **`None`**.
- `consultar_cep(cep)` — a API de verdade:
  1. limpa o CEP; inválido → aviso + `None` **sem ir à internet** (o ViaCEP responderia 400 com uma página HTML — e o `.json()` cairia);
  2. `requests.get(url, timeout=5)` em `https://viacep.com.br/ws/<cep>/json/`;
  3. status fora de 2xx → `None` (`raise_for_status()` ou `status_code` — você escolhe);
  4. corpo com a chave `'erro'` → aviso + `None`;
  5. qualquer `requests.RequestException` (sem rede, timeout, resposta que não é JSON) → aviso + `None`;
  6. senão, devolve **o dicionário que a API respondeu**.

⚠️ Chame sempre `requests.get(...)` (e não `from requests import get`): é por ali que o espião e o sabotador da bancada entram.


In [11]:
# --- código herdado: responde com dados velhos, trava sem timeout, cai sem except ---
# TODO: escreva limpar_cep; reescreva consultar_cep com a API de verdade; apague as RESPOSTAS_SALVAS

def limpar_cep(texto):
    """Tira '-', '.' e espaços; devolve os 8 dígitos do CEP, ou None se não for CEP válido."""
    texto_limpo = texto.replace('-', '').replace('.', '').strip()
    if len(texto_limpo) == 8 and texto_limpo.isdigit():
        return texto_limpo
    return None


def consultar_cep(cep):
    """Consulta o CEP no ViaCEP e devolve o dicionário do endereço ou None em caso de erro/falha."""
    cep_limpo = limpar_cep(cep)
    if not cep_limpo:
        print(f"Aviso: CEP '{cep}' inválido.")
        return None

    url = f'https://viacep.com.br/ws/{cep_limpo}/json/'
    try:
        resposta = requests.get(url, timeout=5)
        if resposta.status_code != 200:
            print(f"Aviso: A API respondeu com status {resposta.status_code}.")
            return None

        dados = resposta.json()
        if 'erro' in dados and dados['erro'] == 'true':
            print(f"Aviso: CEP '{cep_limpo}' não encontrado.")
            return None

        return dados
    except requests.RequestException as e:
        print(f"Aviso: Falha na conexão/consulta da API: {e}")
        return None


In [12]:
# ✅ CONFIRA — Reclamação 3 (esta mesa de teste vai à internet de verdade)
assert REDE_OK, "sem internet não há como conferir a API — conecte e rode tudo de novo (Restart & Run All)"
assert limpar_cep('20040-020') == '20040020', f"'20040-020' limpo é '20040020' — o seu deu {limpar_cep('20040-020')!r}"
assert limpar_cep(' 01311.000 ') == '01311000', f"ponto e espaços também saem — o seu deu {limpar_cep(' 01311.000 ')!r}"
assert limpar_cep('2004002O') is None, "'2004002O' tem a letra O no fim: não é CEP → None"
assert limpar_cep('123') is None, "'123' não tem 8 dígitos → None"

# 🕵️ o espião deixa passar e anota
with espiao_da_rede() as chamadas:
    rio = consultar_cep('20040-020')
    fantasma = consultar_cep('99999999')
    invalido = consultar_cep('2004002O')

assert len(chamadas) == 2, (f"esperava 2 idas à internet (20040020 e 99999999) — a sua fez {len(chamadas)}. "
                            "Menos que 2: as RESPOSTAS_SALVAS ainda respondem no lugar da API. "
                            "Mais que 2: o CEP inválido foi à rede.")
assert all('timeout' in c['kwargs'] for c in chamadas), "requests.get sem timeout: com o wi-fi ruim, o caixa trava para sempre"
assert '/ws/20040020/json' in chamadas[0]['url'], f"a URL usa o CEP limpo: .../ws/20040020/json/ — a sua foi {chamadas[0]['url']!r}"
resposta_de_hoje = chamadas[0]['resposta'].json()
assert rio == resposta_de_hoje, f"o seu consultar_cep tem que devolver o que a API respondeu hoje: {resposta_de_hoje.get('logradouro')!r}"
assert fantasma is None, f"o ViaCEP respondeu 200 com {{'erro': 'true'}} — CEP inexistente devolve None. O seu deu {fantasma}"
assert invalido is None, "texto que não é CEP devolve None"
print(f"A API diz hoje: 20040-020 = {rio['logradouro']}, {rio['bairro']} — o dublê dizia Avenida Rio Branco.")

# 🔌 o sabotador puxa o cabo, deixa a API lenta, tira a API do ar
for problema in ('cabo', 'lenta', '503'):
    with rede_fora(problema) as tentativas:
        resultado = consultar_cep('01311000')
    assert len(tentativas) == 1, f"com a rede '{problema}', o seu consultar_cep nem tentou a API (RESPOSTAS_SALVAS ainda aí?)"
    assert resultado is None, f"rede '{problema}': devolva None com aviso — o seu deu {resultado}"
print("Tudo certo! Reclamação 3 fechada — o endereço vem da API de hoje, e a rede caindo não derruba o caixa. ✅")


Aviso: CEP '99999999' não encontrado.
Aviso: CEP '2004002O' inválido.
A API diz hoje: 20040-020 = Praça Pio X, Centro — o dublê dizia Avenida Rio Branco.
Aviso: Falha na conexão/consulta da API: cabo puxado (simulado pelo CONFIRA)
Aviso: Falha na conexão/consulta da API: a API demorou demais (simulado pelo CONFIRA)
Aviso: A API respondeu com status 503.
Tudo certo! Reclamação 3 fechada — o endereço vem da API de hoje, e a rede caindo não derruba o caixa. ✅


### Reclamação 4 — "O mesmo cliente é consultado toda vez — e o CEP da Dona Marta nunca mais funcionou" (Aula 17)

O colega até tentou um **cache**: um JSON no disco com os CEPs já consultados, para não perguntar duas vezes a mesma coisa (o ViaCEP é gratuito — e bloqueia quem abusa). Mas o cache herdado tem três problemas:

- funcionou na sexta e **quebrou no sábado**: no primeiro dia o arquivo não existia; no segundo, o `carregar_cache` tentou ler o arquivo com `json.loads` — que recebe **texto**, não arquivo;
- a chave é o CEP **como o cliente digitou**: `22070-011` e `22070011` viram dois clientes, duas consultas;
- ele **guarda o `None`**: o wi-fi caiu às 19h quando a Dona Marta pediu, o `None` foi para o cache — e dali em diante o cache respondeu "não existe" para sempre, **mesmo com a internet de volta**.

🔧 **Conserte** `carregar_cache(nome)` e `buscar_endereco(cep, nome_cache)` (o `salvar_cache` está certo):

- `carregar_cache` — `json.load`; arquivo **ausente** ou **corrompido** → cache vazio `{}`, sem traceback.
- `buscar_endereco` — limpa o CEP (inválido → `None`, sem tocar em cache nem rede); se o CEP limpo está no cache, devolve **de lá** (sem internet!); senão, `consultar_cep` e **só guarda no cache o que veio de verdade** — `None` não entra.


In [13]:
# --- código herdado: quebra no segundo dia, duplica CEPs, guarda o None ---
# TODO: conserte carregar_cache e buscar_endereco (salvar_cache está certo)

def carregar_cache(nome):
    """Lê o cache de CEPs do disco; sem arquivo ou corrompido, devolve dicionário vazio."""
    try:
        with open(nome, 'r', encoding='utf-8') as arq:
            return json.load(arq)
    except (FileNotFoundError, json.JSONDecodeError):
        return {}


def salvar_cache(nome, cache):
    """Grava o cache de CEPs no disco."""
    with open(nome, 'w', encoding='utf-8') as arq:
        json.dump(cache, arq, indent=4, ensure_ascii=False)


def buscar_endereco(cep, nome_cache):
    """Devolve o endereço do CEP — do cache se já consultado, senão da API (sem salvar None)."""
    cep_limpo = limpar_cep(cep)
    if not cep_limpo:
        print(f"Aviso: CEP '{cep}' é inválido para busca.")
        return None

    cache = carregar_cache(nome_cache)
    if cep_limpo in cache:
        return cache[cep_limpo]

    dados = consultar_cep(cep_limpo)
    if dados is not None:
        cache[cep_limpo] = dados
        salvar_cache(nome_cache, cache)

    return dados


In [14]:
# ✅ CONFIRA — Reclamação 4 (vai à internet de verdade)
assert REDE_OK, "sem internet não há como conferir a API — conecte e rode tudo de novo (Restart & Run All)"
CACHE_TESTE = 'cache_teste_v3.json'
apagar(CACHE_TESTE)

with espiao_da_rede() as chamadas:
    a = buscar_endereco('22070-011', CACHE_TESTE)
    b = buscar_endereco('22070011', CACHE_TESTE)
    c = buscar_endereco(' 22070.011 ', CACHE_TESTE)
assert len(chamadas) == 1, f"o mesmo CEP três vezes (com hífen, sem hífen, com ponto) = UMA ida à internet — a sua fez {len(chamadas)}"
assert a is not None and a == b == c == chamadas[0]['resposta'].json(), "as três consultas devolvem o endereço que a API respondeu"
with open(CACHE_TESTE, 'r', encoding='utf-8') as arq:
    no_disco = json.load(arq)
assert list(no_disco) == ['22070011'], f"a chave do cache é o CEP limpo — o seu arquivo tem {list(no_disco)}"

# 🔌 19h: o wi-fi cai
with rede_fora('cabo') as tentativas:
    conhecido = buscar_endereco('22070011', CACHE_TESTE)
    dona_marta = buscar_endereco('01311000', CACHE_TESTE)
assert conhecido == a, "sem internet, o cliente CONHECIDO continua sendo atendido — pelo cache"
assert len(tentativas) == 1, f"só o CEP novo podia tentar a rede — houve {len(tentativas)} tentativa(s): o conhecido saiu do cache?"
assert dona_marta is None, "sem internet e fora do cache: None, com aviso"
with open(CACHE_TESTE, 'r', encoding='utf-8') as arq:
    no_disco = json.load(arq)
assert '01311000' not in no_disco, "o None da Dona Marta foi parar no cache — agora ela 'não existe' para sempre"

# 🌐 19h15: a internet volta
with espiao_da_rede() as chamadas:
    dona_marta = buscar_endereco('01311000', CACHE_TESTE)
assert len(chamadas) == 1 and dona_marta == chamadas[0]['resposta'].json(), "com a internet de volta, o CEP da Dona Marta tem que ir à API e funcionar"

# 📝 alguém abriu o cache no Bloco de Notas...
with open(CACHE_TESTE, 'w', encoding='utf-8') as arq:
    arq.write('{"22070011": {"cep": "22070-0')
with espiao_da_rede() as chamadas:
    depois = buscar_endereco('22070011', CACHE_TESTE)
assert depois == a and len(chamadas) == 1, "cache corrompido vira cache vazio — consulta de novo, sem traceback"

with espiao_da_rede() as chamadas:
    lixo = buscar_endereco('abc', CACHE_TESTE)
assert lixo is None and chamadas == [], "'abc' não é CEP: None, sem ir à rede"
apagar(CACHE_TESTE)
print("Tudo certo! Reclamação 4 fechada — pergunta uma vez, lembra para sempre, e não guarda o que não achou. ✅")


Aviso: Falha na conexão/consulta da API: cabo puxado (simulado pelo CONFIRA)
Aviso: CEP 'abc' é inválido para busca.
Tudo certo! Reclamação 4 fechada — pergunta uma vez, lembra para sempre, e não guarda o que não achou. ✅


### Reclamação 5 — "R$ 50,00 cravados pagou frete, Paraty saiu sem rua, e a fila só tem o último pedido" (Aulas 14–15 + tudo junto)

Três bugs de pedido, um de cada aula:

- **Frete.** A regra é *grátis **a partir de** R$ 50,00* — e o herdado usa `>`. O cliente de R$ 50,00 cravados pagou R$ 5,00.
- **CEP de cidade inteira.** Cidades pequenas têm **um CEP só**: o ViaCEP **acha** Paraty (`23970-000`), mas com `logradouro` **vazio** — a consulta deu certo e mesmo assim **não serve para entrega**. Isso o dublê nunca mostraria: é dado de verdade. O herdado montou `', 10 -  - Paraty/RJ'` e mandou o motoboy.
- **Fila.** O herdado grava cada pedido com `'w'` — e a fila fica só com o último. Trocar por `'a'`, como no CSV, **não resolve**: dois objetos JSON colados não são um JSON. A fila é uma **lista** no arquivo: carrega, acrescenta, salva.

🔧 **Conserte** as três funções:

- `montar_endereco(dados, numero)` → `'<logradouro>, <numero> - <bairro> - <cidade>/<UF>'`; sem `logradouro` (CEP de cidade inteira) → **`None`**.
- `montar_pedido(cliente, itens, cep, numero, endereco_manual)` — `itens` chega como lista de `(produto, qtd)`; cada um vira `{'produto', 'qtd', 'preco'}` com o preço do **`cardapio` carregado do disco** (não da constante!); produto fora do cardápio é **ignorado com aviso**; `subtotal`, `taxa` (0.0 **a partir de** `FRETE_GRATIS`, senão `TAXA_ENTREGA`) e `total`; `endereco` via `buscar_endereco(cep, ARQUIVO_CACHE)` + `montar_endereco` — e, se qualquer um dos dois devolver `None`, vale o `endereco_manual` (degradação elegante). Devolve o dicionário com as chaves `cliente`, `itens`, `subtotal`, `taxa`, `total` e `endereco`.
- `enfileirar(nome, pedido)` — lê a fila (lista) do disco — ausente ou corrompida vira `[]` —, acrescenta o pedido, salva (`indent=4`, `ensure_ascii=False`) e devolve **quantos pedidos há na fila**.

**Mesa de teste** — Ana: 2 × 20.00 + 1 × 10.00 = **50.00** → frete grátis, total **50.00**. Bia: 1 × 12.00 + 2 × 8.00 = 28.00 + 5.00 = **33.00** (Paraty: endereço à mão). Caio: 1 × 29.90 (o `X-Tudo` some) + 5.00 = **34.90** (CEP inexistente: endereço à mão). Fila: 50.00 + 33.00 + 34.90 = **117.90**.


In [15]:
# --- código herdado: frete com >, endereço sem rua, fila que se apaga ---
# TODO: conserte montar_endereco, montar_pedido e enfileirar

def montar_endereco(dados, numero):
    """Monta a linha de entrega: logradouro, número - bairro - cidade/UF."""
    if not dados or not dados.get('logradouro'):
        return None
    return f"{dados['logradouro']}, {numero} - {dados['bairro']} - {dados['localidade']}/{dados['uf']}"


def montar_pedido(cliente, itens, cep, numero, endereco_manual):
    """Monta o pedido de delivery completo lendo o cardápio atual do disco."""
    cardapio_atual = carregar_cardapio(ARQUIVO_CARDAPIO)
    lista_itens = []
    subtotal = 0.0

    for produto, qtd in itens:
        if produto in cardapio_atual:
            preco = cardapio_atual[produto]
            lista_itens.append({'produto': produto, 'qtd': qtd, 'preco': preco})
            subtotal += preco * qtd
        else:
            print(f"Aviso: Produto '{produto}' ignorado pois não existe no cardápio.")

    # Regra do frete: grátis a partir de (>=) FRETE_GRATIS
    if subtotal >= FRETE_GRATIS:
        taxa = 0.0
    else:
        taxa = TAXA_ENTREGA

    dados_cep = buscar_endereco(cep, ARQUIVO_CACHE)
    endereco = montar_endereco(dados_cep, numero)

    # Degradação elegante: usa endereço manual se a API/cache não retornar logradouro
    if endereco is None:
        endereco = endereco_manual

    total = subtotal + taxa

    return {
        'cliente': cliente,
        'itens': lista_itens,
        'subtotal': subtotal,
        'taxa': taxa,
        'total': total,
        'endereco': endereco
    }


def enfileirar(nome, pedido):
    """Coloca o pedido na fila de entregas (lista em JSON) e devolve o total de pedidos na fila."""
    try:
        with open(nome, 'r', encoding='utf-8') as arq:
            fila = json.load(arq)
            if not isinstance(fila, list):
                fila = []
    except (FileNotFoundError, json.JSONDecodeError):
        fila = []

    fila.append(pedido)

    with open(nome, 'w', encoding='utf-8') as arq:
        json.dump(fila, arq, indent=4, ensure_ascii=False)

    return len(fila)

In [16]:
# ✅ CONFIRA — Reclamação 5 (vai à internet de verdade)
assert REDE_OK, "sem internet não há como conferir a API — conecte e rode tudo de novo (Restart & Run All)"
pedido_ana = montar_pedido('Ana Souza', [('X-Salada', 2), ('Batata frita', 1)],
                           '22070-011', '1200', 'não usado')
pedido_bia = montar_pedido('Bia Lima', [('Açaí 300ml', 1), ('Suco de laranja', 2)],
                           '23970-000', '10', 'Rua do Comércio, 10 - Centro Histórico - Paraty/RJ')
pedido_caio = montar_pedido('Caio Reis', [('Combo da casa', 1), ('X-Tudo', 2)],
                            '99999999', '7', 'Rua das Flores, 7 - Copacabana - Rio de Janeiro/RJ')

assert abs(pedido_ana['subtotal'] - 50.00) < 0.005, f"2 × 20.00 + 1 × 10.00 = 50.00 — o seu deu {pedido_ana['subtotal']:.2f}"
assert pedido_ana['taxa'] == 0.0, f"50.00 cravados: frete grátis A PARTIR DE 50.00 — o seu cobrou {pedido_ana['taxa']:.2f} (> ou >=?)"
assert abs(pedido_ana['total'] - 50.00) < 0.005, f"total da Ana: 50.00 — o seu deu {pedido_ana['total']:.2f}"
copa = buscar_endereco('22070011', ARQUIVO_CACHE)
esperado = f"{copa['logradouro']}, 1200 - {copa['bairro']} - {copa['localidade']}/{copa['uf']}"
assert pedido_ana['endereco'] == esperado, f"endereço da Ana pelo CEP: {esperado!r} — o seu deu {pedido_ana['endereco']!r}"

paraty = buscar_endereco('23970000', ARQUIVO_CACHE)
assert paraty is not None and paraty['logradouro'] == '', "a API ACHA Paraty — mas sem rua (CEP de cidade inteira)"
assert montar_endereco(paraty, '10') is None, "CEP sem logradouro não serve para entrega: montar_endereco devolve None"
assert pedido_bia['endereco'] == 'Rua do Comércio, 10 - Centro Histórico - Paraty/RJ', f"Paraty: vale o endereço à mão — o seu deu {pedido_bia['endereco']!r}"
assert abs(pedido_bia['total'] - 33.00) < 0.005, f"28.00 + 5.00 de taxa = 33.00 — o seu deu {pedido_bia['total']:.2f}"

assert pedido_caio['itens'] == [{'produto': 'Combo da casa', 'qtd': 1, 'preco': 29.90}], f"o X-Tudo fica fora do pedido — os itens do Caio: {pedido_caio['itens']}"
assert abs(pedido_caio['total'] - 34.90) < 0.005, f"29.90 + 5.00 = 34.90 — o seu deu {pedido_caio['total']:.2f}"
assert pedido_caio['endereco'] == 'Rua das Flores, 7 - Copacabana - Rio de Janeiro/RJ', "CEP inexistente: vale o endereço à mão"

FILA_TESTE = 'fila_teste_v3.json'
apagar(FILA_TESTE)
tamanhos = [enfileirar(FILA_TESTE, p) for p in (pedido_ana, pedido_bia, pedido_caio)]
assert tamanhos == [1, 2, 3], f"a fila cresce 1, 2, 3 — a sua respondeu {tamanhos}"
with open(FILA_TESTE, 'r', encoding='utf-8') as arq:
    fila = json.load(arq)
assert [p['cliente'] for p in fila] == ['Ana Souza', 'Bia Lima', 'Caio Reis'], "a fila do disco tem os três, na ordem de chegada"
assert abs(sum(p['total'] for p in fila) - 117.90) < 0.005, "50.00 + 33.00 + 34.90 = 117.90"

with open(FILA_TESTE, 'w', encoding='utf-8') as arq:   # 📝 de novo o Bloco de Notas...
    arq.write('[{"cliente": "Ana Souza", "itens": [')
assert enfileirar(FILA_TESTE, pedido_caio) == 1, "fila corrompida recomeça vazia — e o pedido novo entra"
apagar(FILA_TESTE)
print("Tudo certo! Reclamação 5 fechada — frete justo, motoboy no endereço certo, fila que não se apaga. ✅")


Aviso: Produto 'X-Tudo' ignorado pois não existe no cardápio.
Aviso: CEP '99999999' não encontrado.
Tudo certo! Reclamação 5 fechada — frete justo, motoboy no endereço certo, fila que não se apaga. ✅


### Reclamação 6 — "Quero ver o sábado inteiro rodar" · ✅ a mesa de teste da release

O sábado de ponta a ponta, nos **arquivos de verdade** da release:

1. **Promoção.** O gerente edita o `cardapio_v3.json` — X-Burguer a **R$ 15,00** — **sem tocar no código**. O sistema tem que obedecer o arquivo.
2. **Balcão.** Turno aberto, cinco tentativas: 2 X-Burguer (2 × 15.00 = 30.00), 1 Combo (29.90), 3 Açaí (36.00 — e o açaí acaba), mais 1 Açaí (recusado: acabou) e uma Pizza (não existe). Fechamento lido do disco: 30.00 + 29.90 + 36.00 = **95.90**.
3. **Delivery.** Ana pede 2 X-Burguer + 1 Refrigerante: 30.00 + 6.50 = 36.50 + 5.00 = **41.50**. Aí **o wi-fi cai**: Duda (cliente conhecido, CEP no cache) pede 3 X-Salada = **60.00** e recebe o endereço **automático**; Edu (CEP novo) pede 2 Combos = **59.80** e o endereço vai **à mão**. Fila: 41.50 + 60.00 + 59.80 = **161.30**.

Nada para escrever aqui: se a célula terminar em "Release v3.0 conferida", a sua release está de pé. Se não, a mensagem do `assert` diz qual reclamação reabriu.


In [17]:
# ✅ CONFIRA — a mesa de teste completa da release v3.0 (o sábado de promoção)
assert REDE_OK, "sem internet não há como conferir a API — conecte e rode tudo de novo (Restart & Run All)"

# 1. a promoção mora no ARQUIVO
promocao = carregar_cardapio(ARQUIVO_CARDAPIO)
promocao['X-Burguer'] = 15.00
salvar_cardapio(ARQUIVO_CARDAPIO, promocao)
cardapio = carregar_cardapio(ARQUIVO_CARDAPIO)
assert cardapio['X-Burguer'] == 15.00 and CARDAPIO_SEMENTE['X-Burguer'] == 18.50, "a promoção mora no arquivo, não na constante (Reclamação 1)"

# 2. o balcão
estoque_sabado = dict(ESTOQUE_INICIAL)
abrir_turno(ARQUIVO_VENDAS)
registrar_venda(ARQUIVO_VENDAS, cardapio, estoque_sabado, 'X-Burguer', 2)       # 30.00
registrar_venda(ARQUIVO_VENDAS, cardapio, estoque_sabado, 'Combo da casa', 1)   # 29.90
registrar_venda(ARQUIVO_VENDAS, cardapio, estoque_sabado, 'Açaí 300ml', 3)      # 36.00 — acabou
try:
    registrar_venda(ARQUIVO_VENDAS, cardapio, estoque_sabado, 'Açaí 300ml', 1)
    recusou_acai = False
except ValueError as erro:
    print('Recusada ->', erro)
    recusou_acai = True
pizza = registrar_venda(ARQUIVO_VENDAS, cardapio, estoque_sabado, 'Pizza', 1)

assert recusou_acai, "o açaí acabou: a quarta unidade levanta ValueError (Reclamação 2)"
assert pizza is None, "Pizza não está no cardápio: None (Reclamação 2)"
assert estoque_sabado['Açaí 300ml'] == 0 and estoque_sabado['X-Burguer'] == 4, "estoque no fim do turno: Açaí 0, X-Burguer 4 (Reclamação 2)"
dia = fechar_dia(ARQUIVO_VENDAS)
assert abs(dia - 95.90) < 0.005, f"30.00 (promoção!) + 29.90 + 36.00 = 95.90 — o seu fechou em {dia:.2f} (Reclamações 1 e 2)"

# 3. o delivery — e o wi-fi caindo no meio
apagar(ARQUIVO_PEDIDOS)
ana = montar_pedido('Ana Souza', [('X-Burguer', 2), ('Refrigerante lata', 1)],
                    '22070-011', '1200', 'não usado')
with rede_fora('cabo') as tentativas:
    duda = montar_pedido('Duda Alves', [('X-Salada', 3)],
                         '22070011', '45', 'não usado')
    edu = montar_pedido('Edu Ramos', [('Combo da casa', 2)],
                        '20040-020', '3', 'Rua Primeiro de Março, 3 - Centro - Rio de Janeiro/RJ')
for pedido in (ana, duda, edu):
    enfileirar(ARQUIVO_PEDIDOS, pedido)

assert abs(ana['total'] - 41.50) < 0.005, f"2 × 15.00 (promoção) + 6.50 + 5.00 = 41.50 — o seu deu {ana['total']:.2f} (o pedido lê o cardápio do disco? Reclamação 5)"
assert abs(duda['total'] - 60.00) < 0.005 and duda['taxa'] == 0.0, "3 × 20.00 = 60.00, frete grátis (Reclamação 5)"
assert len(tentativas) == 1, f"com o wi-fi fora, só o CEP NOVO (Edu) podia tentar a rede — houve {len(tentativas)} tentativa(s). O pedido usa buscar_endereco, com cache? (Reclamações 4 e 5)"
assert duda['endereco'].startswith(ana['endereco'].split(',')[0] + ', 45 - '), f"Duda é cliente conhecido: endereço automático, pelo cache, mesmo sem internet — o seu deu {duda['endereco']!r} (Reclamação 4)"
assert edu['endereco'] == 'Rua Primeiro de Março, 3 - Centro - Rio de Janeiro/RJ', "CEP novo com o wi-fi fora: endereço à mão (Reclamação 5)"
assert abs(edu['total'] - 59.80) < 0.005, "2 × 29.90 = 59.80, frete grátis (Reclamação 5)"

with open(ARQUIVO_PEDIDOS, 'r', encoding='utf-8') as arq:
    fila = json.load(arq)
assert [p['cliente'] for p in fila] == ['Ana Souza', 'Duda Alves', 'Edu Ramos'], "a fila do disco tem os três pedidos do sábado (Reclamação 5)"
assert abs(sum(p['total'] for p in fila) - 161.30) < 0.005, f"41.50 + 60.00 + 59.80 = 161.30 — a fila do disco soma {sum(p['total'] for p in fila):.2f}"
print(f"{LANCHONETE}: balcão R$ {dia:.2f} · delivery R$ {sum(p['total'] for p in fila):.2f} · {len(fila)} entregas na fila.")
print("Release v3.0 conferida! Seis reclamações fechadas: o sistema lembra, acha o cliente na API de verdade e não cai. 🚚✅")


Recusada -> Estoque insuficiente para 'Açaí 300ml'. Disponível: 0.
Aviso: Produto 'Pizza' não encontrado no cardápio.
Aviso: Falha na conexão/consulta da API: cabo puxado (simulado pelo CONFIRA)
Provadelli's : balcão R$ 95.90 · delivery R$ 161.30 · 3 entregas na fila.
Release v3.0 conferida! Seis reclamações fechadas: o sistema lembra, acha o cliente na API de verdade e não cai. 🚚✅


## A defesa oral (sorteio)

No dia do checkpoint, alunos sorteados apresentam a própria release em **3 minutos** e respondem a uma modificação ao vivo. Na versão B, a pergunta começa sempre pelo código herdado:

- *"Mostre a linha do herdado que fazia o fechamento mostrar só a última venda. Por que, depois de trocar o `'w'` pelo `'a'`, ele **quebrou**?"*
- *"Por que a fila de pedidos não se conserta trocando `'w'` por `'a'`, como o CSV?"*
- *"Muda o frete grátis para R$ 60,00. Qual pedido da mesa da Reclamação 5 muda de total? E da Reclamação 6?"*
- *"Puxa o wi-fi agora e faz um pedido para um CEP que já está no cache e outro que não está. O que cada cliente recebe — e em que linha isso é decidido?"*
- *"O dublê dizia Avenida Rio Branco; a API diz outra coisa. Se amanhã o ViaCEP mudar o nome dessa rua de novo, **qual assert quebra**?"*
- *"Por que o cache não guarda o `None`?"*

Pode usar IA para estudar? Pode. Mas o código é seu: **quem não explica e não altera o próprio código, não fecha o checkpoint.** E a versão B tem um agravante: o rascunho do colega **também** veio de IA — e foi ao ar sem ninguém ler. O caderno de reclamações é o preço disso.

## Entrega

- Este notebook completo, com **Restart & Run All** passando do início ao fim **com internet**;
- todas as células ✅ CONFIRA terminando em "Tudo certo" (e a última em "Release v3.0 conferida");
- nenhuma célula terminando em traceback — **essa é a alma da v3.0**;
- os arquivos `cardapio_v3.json`, `vendas_v3.csv`, `cache_cep_v3.json` e `pedidos_v3.json` gerados pelo próprio notebook.

Próxima parada: **Aula 19 — Recursividade** (o combo que contém combos) e, no fim do semestre, a **v4.0 — "O Relatório"**, rodando sobre os dados que você gerou o ano inteiro.
